In [1]:
import os
import pandas as pd

raw_path = "../data/raw/accepted_2007_to_2018Q4.csv"

df = pd.read_csv(raw_path, low_memory=False)

print("Original shape:", df.shape)
df.head()

Original shape: (2260701, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
positive_statuses = [
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Charged Off"
]

negative_statuses = [
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid"
]

valid_statuses = positive_statuses + negative_statuses

df = df[df["loan_status"].isin(valid_statuses)].copy()

df["default"] = df["loan_status"].isin(positive_statuses).astype(int)

print("Shape after selecting resolved loans:", df.shape)
print("\nTarget distribution:")
print(df["default"].value_counts())
print("\nTarget proportions:")
print(df["default"].value_counts(normalize=True))

Shape after selecting resolved loans: (1348099, 152)

Target distribution:
default
0    1078739
1     269360
Name: count, dtype: int64

Target proportions:
default
0    0.800193
1    0.199807
Name: proportion, dtype: float64


In [3]:
sample_size = min(100000, len(df))

df = df.sample(n=sample_size, random_state=42).copy()

print("Sample shape:", df.shape)
print(df["default"].value_counts(normalize=True))

Sample shape: (100000, 152)
default
0    0.79869
1    0.20131
Name: proportion, dtype: float64


In [4]:
drop_cols = [
    "id",
    "member_id",
    "url",
    "title",
    "zip_code",
    "emp_title",
    "desc"
]

leakage_cols = [
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_amnt"
]

post_event_cols = [
    col for col in df.columns
    if "hardship" in col.lower()
    or "settlement" in col.lower()
]

df = df.drop(
    columns=drop_cols + leakage_cols + post_event_cols,
    errors="ignore"
)

# Remove the original status now that the target has been created.
df = df.drop(columns=["loan_status"], errors="ignore")

print("Shape after dropping unnecessary columns:", df.shape)

Shape after dropping unnecessary columns: (100000, 118)


In [5]:
missing_rate = df.isnull().mean().sort_values(ascending=False)

print("Columns with the highest missing rates:")
print(missing_rate.head(15))

high_missing_cols = missing_rate[missing_rate > 0.70].index.tolist()

df = df.drop(columns=high_missing_cols)

print("\nDropped columns:", high_missing_cols)
print("Shape after missing-value filtering:", df.shape)

Columns with the highest missing rates:
next_pymnt_d                                  0.99773
orig_projected_additional_accrued_interest    0.99737
deferral_term                                 0.99590
payment_plan_start_date                       0.99590
sec_app_mths_since_last_major_derog           0.99532
sec_app_revol_util                            0.98668
sec_app_earliest_cr_line                      0.98651
sec_app_inq_last_6mths                        0.98651
sec_app_fico_range_high                       0.98651
sec_app_chargeoff_within_12_mths              0.98651
sec_app_mort_acc                              0.98651
sec_app_fico_range_low                        0.98651
sec_app_collections_12_mths_ex_med            0.98651
revol_bal_joint                               0.98651
sec_app_open_act_il                           0.98651
dtype: float64

Dropped columns: ['next_pymnt_d', 'orig_projected_additional_accrued_interest', 'deferral_term', 'payment_plan_start_date', 'sec_app_m

In [6]:
keep_cols = [
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "dti",
    "delinq_2yrs",
    "fico_range_low",
    "fico_range_high",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "application_type",
    "mort_acc",
    "pub_rec_bankruptcies",
    "default"
]

available_cols = [col for col in keep_cols if col in df.columns]
df = df[available_cols].copy()

if "default" not in df.columns:
    raise ValueError("Target column 'default' is missing.")

print("Selected shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Selected shape: (100000, 25)

Columns:
['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'purpose', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'application_type', 'mort_acc', 'pub_rec_bankruptcies', 'default']


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,application_type,mort_acc,pub_rec_bankruptcies,default
23125,23000.0,36 months,6.24,702.21,A,A2,1 year,RENT,150000.0,Source Verified,...,0.0,9.0,0.0,26479.0,37.2,31.0,Individual,0.0,0.0,0
168272,10000.0,60 months,18.25,255.30,E,E1,< 1 year,RENT,53000.0,Source Verified,...,0.0,9.0,0.0,4784.0,52.0,18.0,Individual,0.0,0.0,1
1684162,25000.0,36 months,7.99,783.30,A,A5,3 years,MORTGAGE,140000.0,Source Verified,...,2.0,7.0,0.0,53593.0,15.9,16.0,Individual,2.0,0.0,0
1987101,5600.0,36 months,7.99,175.46,A,A4,10+ years,RENT,45000.0,Not Verified,...,1.0,12.0,0.0,4105.0,20.4,16.0,Individual,0.0,0.0,1
1701151,17450.0,36 months,14.99,604.83,C,C4,10+ years,RENT,48000.0,Verified,...,0.0,10.0,0.0,6554.0,72.8,17.0,Individual,0.0,0.0,0


In [7]:
print("Overall default rate:")
print(df["default"].mean())

print("\nDefault rate by loan grade:")
if "grade" in df.columns:
    print(df.groupby("grade", observed=True)["default"].mean().sort_values())

print("\nDefault rate by loan purpose:")
if "purpose" in df.columns:
    print(df.groupby("purpose", observed=True)["default"].mean().sort_values())

print("\nDefault rate by annual income:")
if df["annual_inc"].nunique() > 1:
    print(
        df.groupby(
            pd.qcut(df["annual_inc"], 5, duplicates="drop"),
            observed=True
        )["default"].mean()
    )

print("\nDefault rate by interest rate:")
if df["int_rate"].nunique() > 1:
    print(
        df.groupby(
            pd.qcut(df["int_rate"], 5, duplicates="drop"),
            observed=True
        )["default"].mean()
    )

Overall default rate:
0.20131

Default rate by loan grade:
grade
A    0.059722
B    0.133405
C    0.228254
D    0.305619
E    0.386435
F    0.439560
G    0.516340
Name: default, dtype: float64

Default rate by loan purpose:
purpose
wedding               0.129944
car                   0.160190
credit_card           0.170808
major_purchase        0.170952
home_improvement      0.180240
vacation              0.195936
educational           0.212121
debt_consolidation    0.213311
other                 0.214104
house                 0.217712
moving                0.230254
medical               0.231356
renewable_energy      0.287879
small_business        0.289403
Name: default, dtype: float64

Default rate by annual income:
annual_inc
(-0.001, 42000.0]        0.236996
(42000.0, 57000.0]       0.217275
(57000.0, 74000.0]       0.206216
(74000.0, 100000.0]      0.181134
(100000.0, 7000000.0]    0.162953
Name: default, dtype: float64

Default rate by interest rate:
int_rate
(5.308999999999999, 

In [8]:
print("Missing values by column:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nFinal shape:", df.shape)
print("\nTarget distribution:")
print(df["default"].value_counts())

os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/cleaned_data.csv", index=False)

print("\nSaved cleaned_data.csv")

Missing values by column:
emp_length              5670
mort_acc                3771
pub_rec_bankruptcies     125
revol_util                72
dti                       25
open_acc                   4
inq_last_6mths             4
delinq_2yrs                4
pub_rec                    4
total_acc                  4
dtype: int64

Final shape: (100000, 25)

Target distribution:
default
0    79869
1    20131
Name: count, dtype: int64

Saved cleaned_data.csv
